# Phase 2: Multi-Representation Multilingual Normalization & Address Parsing
## Amazon ML Challenge 2026 — Multilingual Business Entity Resolution

This notebook executes the full Phase 2 pipeline:
1. **Step 2.1 (Multi-Representation Name Normalizer):** Extracts `name_clean`, `name_core`, `legal_form`, `name_tokens`, `name_acronym`, and `name_phonetic`.
2. **Step 2.2 (Structured Address Parser):** Extracts `addr_clean`, `postal_clean`, `addr_unit_num`, `addr_digits`, and `addr_tokens`.
3. **Step 2.3 (High-Throughput Streaming Preprocessor):** Enriches all 24.2M records across US, India, and France into fast Parquet files with peak RAM < 250 MB.
4. **Step 2.4 (Normalization Ablation & Collision Diagnostics):** Measures empirical True Match Recall vs Target Bucket Collision Rates.

In [ ]:
# Cell 1: Environment & Module Setup
import os
import sys
import time
import json
from pathlib import Path
import polars as pl

# Locate project root
current_path = Path.cwd().resolve()
while current_path.parent != current_path:
    if (current_path / "dataset").exists() or (current_path / "code").exists():
        project_root = current_path
        break
    current_path = current_path.parent
else:
    project_root = Path.cwd().resolve()

ber_code_dir = project_root / "code" / "business_entity_resolution"
if str(ber_code_dir) not in sys.path:
    sys.path.insert(0, str(ber_code_dir))

print(f"Project Root: {project_root}")
print(f"BER Code Dir: {ber_code_dir}")

from src.config import PROCESSED_DIR, DATASET_DIR, TRAIN_DIR
from src.multilingual_normalizer import normalize_business_name_record
from src.address_parser import parse_address_record
from src.preprocess_datasets import run_full_preprocessing
from src.ablation_normalization import evaluate_normalization_ablation

print("All Phase 2 modules imported successfully!")

In [ ]:
# Cell 2: Unit Testing Normalizer & Address Parser on Multilingual Examples
sample_entities = [
    ("Tata Consultancy Services Pvt. Ltd.", "Plot 42-B, Electronic City, Bengaluru 560100", "India"),
    ("L.L.Bean Inc.", "15 Main Street, Suite 200, Freeport, ME 04032", "US"),
    ("Societe Generale S.A.R.L.", "29 Boulevard Haussmann, 75009 Paris", "France"),
    ("Laxmi Jewellers & Sons", "Shop No 14, MG Road, Indiranagar, Bangalore 560038", "India"),
]

for name, addr, cty in sample_entities:
    print(f"\n{'='*70}")
    print(f"INPUT: {name} | {addr} [{cty}]")
    name_res = normalize_business_name_record(name, cty)
    addr_res = parse_address_record(addr, cty)
    print("--- NAME REPRESENTATIONS ---")
    for k, v in name_res.items():
        print(f"  {k:<16}: {v}")
    print("--- ADDRESS REPRESENTATIONS ---")
    for k, v in addr_res.items():
        print(f"  {k:<16}: {v}")

In [ ]:
# Cell 3: Step 2.3 — Run Streaming Multi-Representation Preprocessing across all 24.2M records
# Note: Uses low-memory streaming PyArrow writer (< 250 MB peak RAM)
run_full_preprocessing(force_recompute=True)

In [ ]:
# Cell 4: Step 2.4 — Normalization Ablation & Target Index Collision Benchmark
ablation_results = evaluate_normalization_ablation(sample_size=100000)

In [ ]:
# Cell 5: Inspect Enriched Sample from Processed Parquet
df_sample = pl.read_parquet(PROCESSED_DIR / "train_source1_cleaned.parquet").head(5)
print("Sample Enriched Records in train_source1_cleaned.parquet:")
print(df_sample.glimpse())